In [1]:
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import Image
from cv_bridge import CvBridge
import cv2

from vision_interfaces.srv import GetDist

import numpy as np
import pyrealsense2 as rs

In [2]:
import time

test_depth = None
test_color = None

class RealsenseCamera(Node):
    def __init__(self, camera='camera'):
        super().__init__('realsense_camera')
        self.publisher_rgb = self.create_publisher(
            Image, f'/real/{camera}/rgb/aligned_image', 20
        )
        self.publisher_depth = self.create_publisher(
            Image, f'/real/{camera}/depth/aligned_image', 20
        )

        self.publisher_cloud_point = self.create_publisher(
            Image, f'/real/{camera}/depth/aligned_image', 20
        )

        self.get_dist_srv = self.create_service(GetDist, "get_distance", self.get_distance)

        self.br = CvBridge()
        self.start_rs_pipeline()
        
        timer_period = 0.05 # seconds
        self.frame_timer = self.create_timer(timer_period, self.frame_callback)

        self.frames = None

    def get_distance(self, request, response):
        try:
            req_position = [request.x, request.y]
            response.dist = self.aligned_depth_frame.get_distance(req_position[0], req_position[1])
        except e:
            self.get_logger.error(e)
            
            response.dist = -1.0
            
        return response
        
    def frame_callback(self):
        _measure = time.perf_counter()
        
        _frames = self.pipeline.wait_for_frames()
        aligned_frames = self.depth_align.process(_frames)

        # Get aligned frames
        self.aligned_depth_frame = aligned_frames.get_depth_frame() # aligned_depth_frame is a 640x480 depth image
        color_frame = aligned_frames.get_color_frame()

        depth_image = np.asanyarray(self.aligned_depth_frame.get_data())
        color_image = np.asanyarray(color_frame.get_data())

        _measure_1 = time.perf_counter() - _measure
        
        self.publisher_rgb.publish(
            self.br.cv2_to_imgmsg(color_image, encoding='rgb8')
        )
        self.publisher_depth.publish(
            self.br.cv2_to_imgmsg(depth_image)
        )

        # self.get_logger().info("before publishing: {} - include publishing: {}".format(_measure_1, time.perf_counter() - _measure))
            
    def start_rs_pipeline(self):
        self.pipeline = rs.pipeline() # Create a pipeline
        config = rs.config()

        # Get device product line for setting a supporting resolution
        pipeline_wrapper = rs.pipeline_wrapper(self.pipeline)
        pipeline_profile = config.resolve(pipeline_wrapper)
        device = pipeline_profile.get_device()
        device_product_line = str(device.get_info(rs.camera_info.product_line))

        found_rgb = False
        for s in device.sensors:
            if s.get_info(rs.camera_info.name) == 'RGB Camera':
                found_rgb = True
                break
        if not found_rgb:
            self.get_logger().info("The demo requires Depth camera with Color sensor")

        self.rs_profile = self.pipeline.start(config) # Start streaming

        self.enable_depth_align()

    def enable_depth_align(self):
        # Getting the depth sensor's depth scale (see rs-align example for explanation)
        depth_sensor = self.rs_profile.get_device().first_depth_sensor()
        depth_scale = depth_sensor.get_depth_scale()
        self.get_logger().info("Depth Scale is: {}".format(depth_scale))
        
        # We will be removing the background of objects more than
        #  clipping_distance_in_meters meters away
        clipping_distance_in_meters = 1 #1 meter
        clipping_distance = clipping_distance_in_meters / depth_scale
        
        # Create an align object
        # rs.align allows us to perform alignment of depth frames to others frames
        # The "align_to" is the stream type to which we plan to align depth frames.
        align_to = rs.stream.color
        self.depth_align = rs.align(align_to)


In [3]:
rclpy.init()

In [4]:
minimal_publisher = RealsenseCamera()

[INFO] [1764851179.921311295] [realsense_camera]: Depth Scale is: 0.0010000000474974513


In [6]:
minimal_publisher.aligned_depth_frame

<pyrealsense2.frame Z16 #123 @1764851184454.287109>

In [5]:
%timeit

# for _ in range(5):
rclpy.spin(minimal_publisher)


KeyboardInterrupt: 

In [ ]:

# Destroy the node explicitly
# (optional - otherwise it will be done automatically
# when the garbage collector destroys the node object)
minimal_publisher.destroy_node()
rclpy.shutdown()